# 01 – Data Loading and Initial Inspection

This notebook loads and performs initial inspection of the chosen  Cell2Cell churn dataset. The goal is to understand structure, data types, missing values, and potential preprocessing requirements

### Load Data 

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/cell2celltrain.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (51047, 58)


,CustomerID,Churn,MonthlyRevenue,MonthlyMinutes,TotalRecurringCharge,DirectorAssistedCalls,OverageMinutes,RoamingCalls,PercChangeMinutes,PercChangeRevenues,...,ReferralsMadeBySubscriber,IncomeGroup,OwnsMotorcycle,AdjustmentsToCreditRating,HandsetPrice,MadeCallToRetentionTeam,CreditRating,PrizmCode,Occupation,MaritalStatus
0,3000002,Yes,24.00,219.0,22.0,0.25,0.0,0.0,-157.0,-19.0,...,0,4,No,0,30,Yes,1-Highest,Suburban,Professional,No
1,3000010,Yes,16.99,10.0,17.0,0.00,0.0,0.0,-4.0,0.0,...,0,5,No,0,30,No,4-Medium,Suburban,Professional,Yes
2,3000014,No,38.00,8.0,38.0,0.00,0.0,0.0,-2.0,0.0,...,0,6,No,0,Unknown,No,3-Good,Town,Crafts,Yes
3,3000022,No,82.28,1312.0,75.0,1.24,0.0,0.0,157.0,8.1,...,0,6,No,0,10,No,4-Medium,Other,Other,No
4,3000026,Yes,17.14,0.0,17.0,0.00,0.0,0.0,0.0,-0.2,...,0,9,No,1,10,No,1-Highest,Other,Professional,Yes


### CustomerID — Unique row level identifier
confirming CustomerId has no predictive value, so it needs to be removed

In [2]:
assert df["CustomerID"].nunique() == len(df), "CustomerID is not fully unique"
print(f"CustomerID: {df['CustomerID'].nunique()} unique values across {len(df)} rows — confirmed identifier, will be dropped in preprocessing.")

CustomerID: 51047 unique values across 51047 rows — confirmed identifier, will be dropped in preprocessing.


## Data Types and Schema

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51047 entries, 0 to 51046
Data columns (total 58 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CustomerID                 51047 non-null  int64  
 1   Churn                      51047 non-null  object 
 2   MonthlyRevenue             50891 non-null  float64
 3   MonthlyMinutes             50891 non-null  float64
 4   TotalRecurringCharge       50891 non-null  float64
 5   DirectorAssistedCalls      50891 non-null  float64
 6   OverageMinutes             50891 non-null  float64
 7   RoamingCalls               50891 non-null  float64
 8   PercChangeMinutes          50680 non-null  float64
 9   PercChangeRevenues         50680 non-null  float64
 10  DroppedCalls               51047 non-null  float64
 11  BlockedCalls               51047 non-null  float64
 12  UnansweredCalls            51047 non-null  float64
 13  CustomerCareCalls          51047 non-null  flo

The dataset contains `58` features, including numerical and categorical variables. Most features are complete, with only a small number containing missing values. The target variable, `Churn`, is stored as `'Yes'`/`'No'` and will be converted to a binary format during preprocessing.

## Missing Values (NaN)
Display only features with at least one missing value

In [4]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing_df = missing.reset_index()
missing_df.columns = ['Feature', 'Missing Count']

missing_df

,Feature,Missing Count
0,AgeHH1,909
1,AgeHH2,909
2,PercChangeMinutes,367
3,PercChangeRevenues,367
4,MonthlyRevenue,156
5,MonthlyMinutes,156
6,TotalRecurringCharge,156
7,DirectorAssistedCalls,156
8,OverageMinutes,156
9,RoamingCalls,156


## Hidden Missing Values — `'Unknown'` Strings

Some categorical features encode missing values as `'Unknown'`, which are not detected by standard missing value checks. These are identified here and will be converted to `NaN` during preprocessing.

In [5]:
unknown_counts = {}

for col in df.select_dtypes(include='object'):
    count = (df[col] == 'Unknown').sum()
    if count > 0:
        unknown_counts[col] = count

unknown_df = pd.DataFrame(
    unknown_counts.items(),
    columns=['Feature', 'Unknown Count']
)

unknown_df['Unknown (%)'] = (unknown_df['Unknown Count'] / len(df) * 100).round(2)

unknown_df.sort_values(by='Unknown (%)', ascending=False)



,Feature,Unknown Count,Unknown (%)
1,HandsetPrice,28982,56.78
2,MaritalStatus,19700,38.59
0,Homeownership,17060,33.42


The scale of hidden missingness is substantial: `HandsetPrice` has over half its values missing (56.78%), while `MaritalStatus` (38.59%) and `Homeownership` (33.42%) are also significantly affected. These will require careful handling during preprocessing.

## IncomeGroup Interpretation

IncomeGroup = 0 was initially suspected to represent missing or unknown income. However, comparison of churn rates shows similar behaviour to other income groups, indicating that it functions as a valid category rather than a missing value indicator.

In [8]:
zero_rate = (df[df["IncomeGroup"] == 0]["Churn"] == "Yes").mean()
nonzero_rate = (df[df["IncomeGroup"] != 0]["Churn"] == "Yes").mean()

print("Churn rate (IncomeGroup = 0):", round(zero_rate, 3))
print("Churn rate (IncomeGroup ≠ 0):", round(nonzero_rate, 3))

Churn rate (IncomeGroup = 0): 0.302
Churn rate (IncomeGroup ≠ 0): 0.283


## Summary Statistics

In [ ]:
df.describe()

Several numerical features exhibit wide ranges and potential outliers. For example, `MonthlyRevenue`, `PercChangeMinutes`, and `OverageMinutes` show extreme values, suggesting skewed distributions. These characteristics will be addressed during preprocessing. 

Some features contain negative values (e.g., `MonthlyRevenue`, `TotalRecurringCharge`), which may represent anomalies or billing adjustments and will be reviewed during preprocessing.

### Categorical Features Summary 

In [ ]:
cat_summary.sort_values(by='Unique Values', ascending=False)

This table summarises categorical features by showing the number of unique values and the most frequent category. Most features have a small number of unique values, indicating simple binary variables, while `ServiceArea` exhibits high cardinality, meaning it contains many distinct categories and may be more complex to model. The most frequent category also highlights dominant values, with `'Unknown'` appearing in some features, suggesting hidden missing data.

## Target Variable — Class Distribution


In [ ]:
churn_counts = df["Churn"].value_counts()
churn_pct = df["Churn"].value_counts(normalize=True).mul(100).round(2)
display(pd.DataFrame({"Count": churn_counts, "Percentage (%)": churn_pct}))

fig, ax = plt.subplots(figsize=(5, 4))
labels = ["No Churn", "Churn"]
values = [churn_counts["No"], churn_counts["Yes"]]
colors = ["steelblue", "tomato"]
bars = ax.bar(labels, values, color=colors)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 300,
            f"{val:,} ({val / len(df) * 100:.1f}%)",
            ha="center", fontsize=10)
ax.set_title("Class Distribution — Churn vs No Churn")
ax.set_ylabel("Count")
ax.set_ylim(0, max(values) * 1.15)
plt.tight_layout()
plt.show()

The dataset is moderately imbalanced, with approximately **71% non-churn** and **29% churn**. This imbalance may bias models towards the majority class, leading to misleadingly high accuracy but poor identification of churners — the class of primary business interest. Therefore, accuracy is not a suitable primary metric; **Recall, F1-score, and ROC-AUC** will be used.

## Summary of Key Findings for Downstream Processing

The initial inspection reveals a dataset that is sufficiently large for this project but requires careful preprocessing before modelling. 

**The key findings are:**

- **Dataset size:** 51,047 labelled records across 58 features. A separate holdout set of 20,000 records is reserved for final evaluation.

- **Class imbalance:** ~71% non-churn vs ~29% churn. This imbalance suggests that accuracy alone is not a suitable evaluation metric.

- **Non-predictive identifier:** `CustomerID` is a unique row-level identifier and does not provide predictive value.

- **NaN missing values:** Several features contain low levels of missing data (generally below 2%).

- **Hidden missing values:** `HandsetPrice`, `MaritalStatus`, and `Homeownership` contain `'Unknown'` values, indicating missing data not captured by standard checks.

- **High-cardinality feature:** `ServiceArea` contains a large number of unique values, making it more complex to encode.

- **Collinearity:** `NewCellphoneUser` and `NotNewCellphoneUser` appear to represent overlapping information.

- **Anomalous values:** Some features contain unexpected values (e.g., negative values in `MonthlyRevenue` and `TotalRecurringCharge`), suggesting potential data quality issues.

- **Skewed distributions:** Several numerical features exhibit skewness and extreme values, which may impact model performance.